# 06 — Marker gene analysis (exploratory)

**Input:** `results/adata/05_spatial.h5ad`  
**Output:** `results/adata/06_markers.h5ad` and marker tables/figures.

Find genes enriched in each Leiden spatial-bin cluster using Scanpy Wilcoxon tests. This is **exploratory**: 56 µm bins may contain multiple cells, QC filters were not applied, spatial bins are correlated, and bin-level p-values should **not** be interpreted as independent-sample confirmatory inference or cell-type identification. No tissue-image overlay is attempted.

In [ ]:
from pathlib import Path
from importlib.metadata import version
import sys
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
import matplotlib.pyplot as plt

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "results/adata/05_spatial.h5ad").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run Step 05 first: results/adata/05_spatial.h5ad not found")
INPUT = ROOT / "results/adata/05_spatial.h5ad"
OUTPUT = ROOT / "results/adata/06_markers.h5ad"
FIG = ROOT / "results/figures"
TAB = ROOT / "results/tables"
for p in (OUTPUT.parent, FIG, TAB):
    p.mkdir(parents=True, exist_ok=True)
SEED = 42
np.random.seed(SEED)
print("Python:", sys.executable)
print("Scanpy:", version("scanpy"), "AnnData:", version("anndata"))
print("Project root:", ROOT)

## 1. Load and validate spatial analysis

In [ ]:
adata = sc.read_h5ad(INPUT)
assert "leiden" in adata.obs and "spatial" in adata.obsm
assert "counts" in adata.layers and sparse.issparse(adata.layers["counts"])
assert "spatial_connectivities" in adata.obsp and "connectivities" in adata.obsp
assert "log1p" in adata.uns, "Expected log1p-normalized X from Step 03"
adata.obs["leiden"] = adata.obs["leiden"].astype("category")
cluster_sizes = adata.obs["leiden"].value_counts().sort_index()
cluster_sizes.rename_axis("leiden").reset_index(name="n_bins").to_csv(
    TAB / "06_cluster_sizes.csv", index=False)
print("AnnData:", adata.shape, "Clusters:", adata.obs["leiden"].nunique())
print("Smallest cluster:", cluster_sizes.min(), "bins")
print("Using log1p-normalized expression in adata.X (not raw counts)")

## 2. Differential expression across spatial-bin clusters

Wilcoxon rank-sum, each Leiden cluster vs all remaining bins, Benjamini–Hochberg correction. This ignores spatial dependence, so adjusted p-values are **descriptive screening statistics** rather than valid tissue-level evidence. The analysis uses **all genes**, not only HVGs.

In [ ]:
KEY = "rank_genes_leiden_wilcoxon"
sc.tl.rank_genes_groups(
    adata, groupby="leiden", method="wilcoxon", reference="rest",
    use_raw=False, n_genes=adata.n_vars, pts=True,
    key_added=KEY, tie_correct=False,
)
print("PASS — Marker ranking calculated for", adata.obs["leiden"].nunique(), "clusters")

## 3. Export complete results and filtered marker shortlist

In [ ]:
frames = []
for cluster in adata.obs["leiden"].cat.categories:
    df = sc.get.rank_genes_groups_df(adata, group=str(cluster), key=KEY)
    df.insert(0, "leiden", str(cluster))
    frames.append(df)
markers = pd.concat(frames, ignore_index=True)
markers.to_csv(TAB / "06_all_marker_statistics.csv", index=False)

# Screening criteria are adjustable, not biological validation.
MIN_LOGFC = 0.5
MAX_ADJ_P = 0.05
TOP_N = 10
filtered = markers.loc[
    (markers["logfoldchanges"] >= MIN_LOGFC)
    & (markers["pvals_adj"] < MAX_ADJ_P)
].copy()
filtered = filtered.sort_values(["leiden", "scores"], ascending=[True, False])
top = filtered.groupby("leiden", sort=False).head(TOP_N).copy()
top.to_csv(TAB / "06_top10_markers_per_cluster.csv", index=False)
summary = pd.DataFrame({
    "leiden": cluster_sizes.index.astype(str),
    "n_bins": cluster_sizes.to_numpy(),
    "n_markers_padj_lt_0_05_logfc_ge_0_5": [
        int((filtered["leiden"] == str(c)).sum()) for c in cluster_sizes.index
    ],
})
summary.to_csv(TAB / "06_marker_cluster_summary.csv", index=False)
print("Total gene–cluster comparisons:", len(markers))
print("Shortlisted markers:", len(top))
print("Clusters with >=1 shortlisted marker:", top["leiden"].nunique())
display(top.head(20))

## 4. Visualize cluster sizes and representative marker expression

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(cluster_sizes.index.astype(str), cluster_sizes.values)
ax.set(xlabel="Leiden cluster", ylabel="Number of spatial bins", title="Spatial-bin cluster sizes")
ax.tick_params(axis="x", rotation=90)
fig.tight_layout()
fig.savefig(FIG / "06_cluster_sizes.png", dpi=170)
plt.show()

# Select one leading marker per cluster, then deduplicate gene names.
representatives = (top.groupby("leiden", sort=False).head(1)["names"].drop_duplicates().tolist())
if representatives:
    # Cap to keep plots readable; full marker table is always exported.
    selected = representatives[:30]
    sc.pl.dotplot(adata, var_names=selected, groupby="leiden", use_raw=False,
                  standard_scale="var", show=False)
    plt.savefig(FIG / "06_marker_dotplot.png", dpi=170, bbox_inches="tight")
    plt.show()
    sc.pl.matrixplot(adata, var_names=selected, groupby="leiden", use_raw=False,
                     standard_scale="var", cmap="viridis", show=False)
    plt.savefig(FIG / "06_marker_heatmap.png", dpi=170, bbox_inches="tight")
    plt.show()
else:
    print("No markers pass screening thresholds; plots skipped. Inspect full table.")

## 5. Spatial expression maps for candidate genes

Plot coordinate-only expression patterns. The full-resolution spatial coordinates have **no verified mapping to the hires tissue PNG**, so no overlay is used.

In [ ]:
CANDIDATES = ["COL1A1", "COL2A1", "ACAN", "SPP1", "BGLAP", "DCN", "LUM", "PTPRC", "PECAM1"]
genes = [g for g in CANDIDATES if g in adata.var_names][:6]
if genes:
    xy = np.asarray(adata.obsm["spatial"])
    fig, axes = plt.subplots(2, 3, figsize=(15, 9), squeeze=False)
    for ax, gene in zip(axes.flat, genes):
        x = adata[:, gene].X
        expr = x.toarray().ravel() if sparse.issparse(x) else np.asarray(x).ravel()
        points = ax.scatter(xy[:, 0], xy[:, 1], c=expr, s=3, cmap="magma", linewidths=0)
        ax.set(title=gene, xlabel="Fullres x (pixels)", ylabel="Fullres y (pixels)")
        ax.invert_yaxis()
        ax.set_aspect("equal", adjustable="box")
        fig.colorbar(points, ax=ax, fraction=0.045, pad=0.02)
    for ax in axes.flat[len(genes):]: ax.axis("off")
    fig.suptitle("Candidate gene expression — spatial coordinates only")
    fig.tight_layout()
    fig.savefig(FIG / "06_candidate_gene_spatial.png", dpi=170)
    plt.show()
else:
    print("No candidate genes found; spatial expression map skipped")

## 6. Save and verify

In [ ]:
adata.uns["step06_settings"] = {
    "method": "wilcoxon", "reference": "rest", "groupby": "leiden",
    "marker_key": KEY, "min_logfoldchange": float(MIN_LOGFC),
    "max_adjusted_p": float(MAX_ADJ_P), "top_n": int(TOP_N),
    "exploratory_only": True, "qc_filters_applied": False,
    "image_registration_verified": False,
}
adata.write_h5ad(OUTPUT, compression="gzip")
check = sc.read_h5ad(OUTPUT)
assert check.shape == adata.shape
assert check.obs["leiden"].equals(adata.obs["leiden"])
assert np.array_equal(check.obsm["spatial"], adata.obsm["spatial"])
assert "spatial_connectivities" in check.obsp and "connectivities" in check.obsp
assert "counts" in check.layers and sparse.issparse(check.layers["counts"])
assert KEY in check.uns
assert (TAB / "06_all_marker_statistics.csv").exists()
print("PASS — 06_marker_analysis completed")
print("AnnData:", check.shape)
print("Leiden clusters:", check.obs["leiden"].nunique())
print("All marker comparisons:", len(markers))
print("Output:", OUTPUT.relative_to(ROOT))

### Next

Review marker genes against trusted skeletal-tissue references before naming spatial domains. **Do not label Leiden clusters as individual cell types based solely on marker rankings.** Next steps may include biological annotation, robustness checks, and optional Tangram integration with a suitable matched scRNA-seq reference.